In [15]:
import csv, json
from pathlib import Path

ifc_mode = False
ifc_path = "/content/Campus.ifc"
output_csv = "door_check_report.csv"
output_html = "door_ampel_dashboard.html"

IDS_RULES = [
    {"id":"fire_rating_required","property":("Pset_DoorCommon","FireRating"), "check":"exists", "severity":"red", "message":"Missing FireRating"},
    {"id":"width_barrier_free","property":("","OverallWidth"), "check":"min_value", "value":900, "severity":"yellow", "message":"Width < 900 mm (barrier-free)"},
    {"id":"height_datatype","property":("","OverallHeight"), "check":"datatype_number", "severity":"yellow", "message":"Height not numeric"},
    {"id":"width_reasonable","property":("","OverallWidth"), "check":"min_value", "value":300, "severity":"red", "message":"Width < 300 mm (implausible)"},
]

SAMPLE_DOORS = [
    {"id":"Door-001","name":"Eingang Nord","OverallWidth":1000,"OverallHeight":2100,"psets":{"Pset_DoorCommon":{"FireRating":"EI60"}},"room":"Lobby"},
    {"id":"Door-002","name":"WC Damen","OverallWidth":780,"OverallHeight":"2.10m","psets":{"Pset_DoorCommon":{}},"room":"Sanitär"},
    {"id":"Door-003","name":"Lager","OverallWidth":600,"OverallHeight":2000,"psets":{"Pset_DoorCommon":{"FireRating":None}},"room":"Lager"},
    {"id":"Door-004","name":"Schranktür","OverallWidth":250,"OverallHeight":1800,"psets":{"Pset_DoorCommon":{}},"room":"Akten"}
]

def safe_get_pset(element, pset_name, prop_name):
    psets = element.get("psets", {})
    p = psets.get(pset_name, {})
    return p.get(prop_name, None)

def normalize_number(value):
    if value is None:
        return None
    if isinstance(value, str):
        s = value.strip().lower()
        if s.endswith("m"):
            try:
                num_m = float(s.replace("m","").strip())
                return int(round(num_m * 1000))
            except:
                return None
        s2 = s.replace(",",".")
        try:
            return int(round(float(s2)))
        except:
            return None
    if isinstance(value, (int,float)):
        if value > 50:
            return int(round(value))
        else:
            return int(round(float(value) * 1000))
    return None

def evaluate_rule_on_element(rule, el):
    issues = []
    prop = rule["property"]
    if prop[0] == "":
        val = el.get(prop[1], None)
    else:
        val = safe_get_pset(el, prop[0], prop[1])
    check = rule["check"]

    if check == "exists":
        if val is None or (isinstance(val, str) and val.strip() == ""):
            issues.append({"rule":rule["id"], "severity":rule["severity"], "message":rule["message"]})

    elif check == "min_value":
        num = normalize_number(val)
        if num is None:
            issues.append({"rule":rule["id"], "severity":"yellow", "message":"Non-numeric value for min_value check"})
        else:
            if num < rule["value"]:
                issues.append({"rule":rule["id"], "severity":rule["severity"], "message":rule["message"]})

    elif check == "datatype_number":
        num = normalize_number(el.get(prop[1], None))
        if num is None:
            issues.append({"rule":rule["id"], "severity":rule["severity"], "message":rule["message"]})

    else:
        issues.append({"rule":rule["id"], "severity":"yellow", "message":"Unknown check "+str(check)})

    return issues

def compute_traffic_light(issues):
    severities = {iss["severity"] for iss in issues}
    if "red" in severities:
        return "red"
    if "yellow" in severities:
        return "yellow"
    return "green"

def parse_ifc_doors(ifc_path):
    try:
        import ifcopenshell
        from ifcopenshell.util.element import get_psets
        model = ifcopenshell.open(ifc_path)
        doors = model.by_type("IfcDoor")
        parsed = []
        for d in doors:
            data = {
                "id": getattr(d,"GlobalId",None),
                "name": getattr(d,"Name",None),
                "OverallWidth": getattr(d,"OverallWidth", None),
                "OverallHeight": getattr(d,"OverallHeight", None),
                "psets": get_psets(d),
                "room": None
            }
            parsed.append(data)
        return parsed
    except Exception as e:
        raise RuntimeError(f"IFC parsing failed: {e}")

def run_check(doors, rules):
    results = []
    for el in doors:
        el_issues = []
        for r in rules:
            res = evaluate_rule_on_element(r, el)
            el_issues.extend(res)
        traffic = compute_traffic_light(el_issues)
        results.append({
            "id": el.get("id"),
            "name": el.get("name"),
            "room": el.get("room"),
            "width": el.get("OverallWidth"),
            "height": el.get("OverallHeight"),
            "issues": el_issues,
            "traffic": traffic
        })
    return results

def write_csv(results, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["DoorID","Name","Room","Width_mm","Height_mm","Traffic","IssuesJSON"])
        for r in results:
            writer.writerow([r["id"], r["name"], r["room"], r["width"], r["height"], r["traffic"], json.dumps(r["issues"], ensure_ascii=False)])

def generate_simple_html(results, path):
    html = []
    html.append("<html><head><meta charset='utf-8'><title>Tür Ampel Dashboard</title></head><body>")
    html.append("<h1>Tür Ampel Dashboard</h1>")
    html.append("<table border='1' cellpadding='8' style='border-collapse:collapse;'>")
    html.append("<tr><th>DoorID</th><th>Name</th><th>Room</th><th>Width</th><th>Height</th><th>Status</th><th>Issues</th></tr>")
    color_map = {"green":"#c6efce","yellow":"#fff2cc","red":"#f4c7c3"}
    for r in results:
        issues_text = "<br>".join([f"{i['severity'].upper()}: {i['message']}" for i in r["issues"]]) or "-"
        color = color_map.get(r["traffic"], "#ffffff")
        html.append(f"<tr style='background:{color}'><td>{r['id']}</td><td>{r['name']}</td><td>{r['room']}</td><td>{r['width']}</td><td>{r['height']}</td><td style='font-weight:bold'>{r['traffic'].upper()}</td><td>{issues_text}</td></tr>")
    html.append("</table></body></html>")
    Path(path).write_text("\n".join(html), encoding="utf-8")
    print(f"HTML Dashboard geschrieben:", path)

def main():
    if ifc_mode:
        try:
            doors = parse_ifc_doors(ifc_path)
            if not doors:
                doors = SAMPLE_DOORS
        except:
            doors = SAMPLE_DOORS
    else:
        doors = SAMPLE_DOORS

    results = run_check(doors, IDS_RULES)
    write_csv(results, output_csv)
    generate_simple_html(results, output_html)
    print("Fertig. Dateien erstellt:", output_csv, output_html)

main()


HTML Dashboard geschrieben: door_ampel_dashboard.html
Fertig. Dateien erstellt: door_check_report.csv door_ampel_dashboard.html


In [16]:
from google.colab import files

files.download("door_check_report.csv")
files.download("door_ampel_dashboard.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>